# Notebook 01: Thu thập dữ liệu

Notebook gồm 4 phần. Các bảng được tính từ dữ liệu trong `data/raw/`; chạy các cell từ trên xuống để cập nhật kết quả.

## 1. Nguồn dữ liệu và lý do chọn Chợ Tốt

Dự án thu thập tin rao bán ô tô đã qua sử dụng trên Chợ Tốt, với phạm vi truy vấn toàn quốc. Nguồn lấy dữ liệu là API danh sách `https://gateway.chotot.com/v1/public/ad-listing` và API chi tiết cùng địa chỉ, thêm `/{list_id}`.

Chợ Tốt được chọn vì phản hồi JSON có sẵn giá, hãng xe, dòng xe, năm sản xuất, kilomet và địa bàn, thuận tiện đưa vào bảng để phân tích và định giá. API chi tiết còn cung cấp mô tả dài để phân tích văn bản. Crawler đọc các trường có cấu trúc nên không cần tách dữ liệu từ giao diện HTML.

Đây là giá rao bán và thông tin do người đăng cung cấp. Tập tin thu thập không mặc nhiên đại diện cho toàn bộ thị trường xe cũ Việt Nam.

## 2. Cách crawler hoạt động

Script [`crawler/data_raw.py`](crawler/data_raw.py) thực hiện các bước:

1. Gọi API với `cg=2010` (ô tô), `condition_ad=1` (đã sử dụng), mỗi trang tối đa 50 tin.
2. Duyệt 11 dải giá tăng dần, phân trang bằng `offset` và kiểm tra danh mục, tình trạng, mã tin, giá.
3. Lưu vào SQLite, chống trùng bằng `list_id` và ghi tiến độ để có thể chạy tiếp.
4. Dừng khi đủ mục tiêu 15.000 tin hoặc hết dữ liệu; xuất CSV, JSONL và báo cáo. Mặc định chờ 1,5 giây giữa các trang, thử lại khi lỗi mạng/máy chủ và chờ khi gặp HTTP 429.

Chia dải giá nhằm tránh giới hạn kết quả của một truy vấn được ghi nhận trong mã crawler. Việc dừng ở số lượng mục tiêu có thể khiến các dải giá cao chưa được duyệt hết; một chiếc xe cũng có thể xuất hiện ở nhiều mã tin.

Lệnh thu thập từ gốc dự án:

```bash
python 01_crawler_and_ml/crawler/data_raw.py --target 15000 --delay 1.5 --out ../../data/raw
```

## 3. Mô tả 35 trường dữ liệu thô

`data/raw/cars.csv` gồm 33 trường lấy từ API và 2 trường truy vết do crawler bổ sung (`source_url`, `fetched_at`). Bảng dưới mô tả đủ 35 trường theo thứ tự trong CSV. Các mã phân loại và giá trị thiếu được giữ nguyên để xử lý ở notebook 02.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display


ROOT = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "requirements.txt").is_file()
        and (folder / "data" / "raw" / "cars.csv").is_file()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Không tìm thấy gốc dự án có data/raw/cars.csv.")

RAW_DIR = ROOT / "data" / "raw"
raw = pd.read_csv(
    RAW_DIR / "cars.csv",
    encoding="utf-8-sig",
    low_memory=False,
    dtype={"list_id": "string", "ad_id": "string"},
)
if raw.empty or raw["list_id"].isna().any() or not raw["list_id"].is_unique:
    raise ValueError("Dữ liệu gốc rỗng hoặc có mã tin thiếu/trùng.")


def show_table(frame):
    display(HTML(frame.to_html(index=False, escape=True, border=0)))


print(f"Dữ liệu thô: {len(raw):,} tin, {len(raw.columns)} trường.")

Dữ liệu thô: 15,000 tin, 35 trường.


In [2]:
field_meanings = {
    "list_id": "Mã tin; khóa chống trùng và nối với mô tả.",
    "ad_id": "Mã quảng cáo do API trả về.",
    "subject": "Tiêu đề tin đăng.",
    "price": "Giá rao bán dạng số, đơn vị đồng Việt Nam.",
    "price_string": "Giá dạng chuỗi hiển thị.",
    "category": "Mã danh mục; 2010 là ô tô.",
    "category_name": "Tên danh mục.",
    "condition_ad": "Mã tình trạng; 1 là đã sử dụng.",
    "condition_ad_name": "Tên tình trạng xe.",
    "carbrand": "Mã hãng xe.",
    "carbrand_name": "Tên hãng xe.",
    "carmodel": "Mã dòng xe.",
    "carmodel_name": "Tên dòng xe.",
    "mfdate": "Năm sản xuất được khai báo.",
    "mileage_v2": "Số kilomet đã đi theo người đăng.",
    "gearbox": "Mã loại hộp số.",
    "fuel": "Mã loại nhiên liệu.",
    "carseats": "Mã nhóm số chỗ ngồi; cần ánh xạ sang số chỗ.",
    "cartype": "Mã kiểu dáng xe.",
    "carcolor": "Mã màu xe.",
    "carorigin": "Mã xuất xứ xe.",
    "region_v2": "Mã tỉnh/thành theo API.",
    "region_name": "Tên tỉnh/thành.",
    "region_name_v3": "Tên địa bàn theo phiên bản v3 của API.",
    "area_name": "Tên quận/huyện hoặc khu vực.",
    "ward_name": "Tên phường/xã.",
    "list_time": "Thời điểm đăng/hiển thị, Unix mili giây.",
    "orig_list_time": "Thời điểm đăng gốc theo API, Unix mili giây.",
    "image": "URL ảnh đại diện.",
    "images": "Danh sách URL ảnh, lưu thành chuỗi JSON.",
    "number_of_images": "Số ảnh của tin theo API.",
    "type": "Mã loại tin; giữ nguyên giá trị API.",
    "status": "Trạng thái tin khi thu thập danh sách.",
    "source_url": "URL truy vấn trả về tin, do crawler bổ sung.",
    "fetched_at": "Thời điểm thu thập theo UTC, dạng ISO 8601.",
}
if len(raw.columns) != 35 or list(raw.columns) != list(field_meanings):
    raise ValueError("Cấu trúc CSV không khớp từ điển 35 trường.")

show_table(pd.DataFrame({
    "STT": range(1, 36),
    "Trường": raw.columns,
    "Ý nghĩa": [field_meanings[column] for column in raw.columns],
}))

STT,Trường,Ý nghĩa
1,list_id,Mã tin; khóa chống trùng và nối với mô tả.
2,ad_id,Mã quảng cáo do API trả về.
3,subject,Tiêu đề tin đăng.
4,price,"Giá rao bán dạng số, đơn vị đồng Việt Nam."
5,price_string,Giá dạng chuỗi hiển thị.
6,category,Mã danh mục; 2010 là ô tô.
7,category_name,Tên danh mục.
8,condition_ad,Mã tình trạng; 1 là đã sử dụng.
9,condition_ad_name,Tên tình trạng xe.
10,carbrand,Mã hãng xe.


## 4. Kết quả cào mô tả

Script [`crawler/descriptions.py`](crawler/descriptions.py) đọc `list_id` từ `cars.csv`, gọi API chi tiết và lưu trường `body` thành `description` khi tin còn hoạt động và nội dung không rỗng. Kết quả lưu trong `descriptions.csv`; `description_status.csv` ghi trạng thái từng mã tin, còn `descriptions_report.json` tổng hợp độ phủ và thời gian cào.

Đợt kiểm tra ngày 28/09/2026 đã xử lý đủ 15.000 mã tin: lấy được **10.547 mô tả (70,31%)**, còn **4.453 tin (29,69%)** trả HTTP 404/410; không còn mã chờ kiểm tra hoặc lỗi chưa xử lý. `removed` chỉ có nghĩa API chi tiết không còn truy cập được, chưa xác định lý do.

Mô tả được cào sau dữ liệu danh sách nên trạng thái tin có thể thay đổi. Notebook 04 nối trái theo `list_id` để giữ các tin thiếu mô tả; nếu dùng tiêu đề thay thế thì cần ghi rõ nguồn văn bản.

In [3]:
descriptions = pd.read_csv(
    RAW_DIR / "descriptions.csv",
    encoding="utf-8-sig",
    dtype="string",
    keep_default_na=False,
)
status = pd.read_csv(
    RAW_DIR / "description_status.csv",
    encoding="utf-8-sig",
    dtype="string",
    keep_default_na=False,
)
report = json.loads((RAW_DIR / "descriptions_report.json").read_text(encoding="utf-8-sig"))

raw_ids = set(raw["list_id"])
ok_ids = set(status.loc[status["status"].eq("ok"), "list_id"])
counts = status["status"].value_counts()
assert status["list_id"].is_unique and set(status["list_id"]) == raw_ids
assert list(descriptions.columns) == ["list_id", "description"]
assert descriptions["list_id"].is_unique and set(descriptions["list_id"]) == ok_ids
assert descriptions["description"].str.strip().ne("").all()
assert status.loc[status["status"].eq("removed"), "http_status"].isin(["404", "410"]).all()
assert report["total_ids"] == len(raw_ids)
assert all(
    report["counts"].get(key, 0) == int(counts.get(key, 0))
    for key in set(counts.index) | set(report["counts"])
)
assert abs(report["description_coverage"] - len(descriptions) / len(raw_ids)) < 1e-9

status_labels = {
    "ok": "Lấy được mô tả",
    "removed": "API trả HTTP 404/410",
    "empty": "Mô tả rỗng",
    "unavailable": "Tin không hoạt động",
    "pending": "Chưa kiểm tra",
    "http_error": "Lỗi HTTP khác",
    "network_error": "Lỗi mạng",
    "invalid_response": "Phản hồi không hợp lệ",
    "rate_limited": "Giới hạn tốc độ",
}
if not set(counts.index).issubset(status_labels):
    raise ValueError("Có trạng thái mô tả chưa được định nghĩa.")

In [4]:
status_counts = counts.reindex(status_labels, fill_value=0)
summary = pd.DataFrame({
    "Kết quả": list(status_labels.values()),
    "Số tin": status_counts.to_numpy(),
    "Tỷ lệ (%)": (status_counts / len(raw_ids) * 100).round(2).to_numpy(),
})
show_table(summary)
coverage = len(descriptions) / len(raw_ids)
print(f"Độ phủ mô tả: {len(descriptions):,}/{len(raw_ids):,} tin ({coverage:.2%}).")
checked_times = pd.to_datetime(status["checked_at"], utc=True).dt.tz_convert("Asia/Ho_Chi_Minh")
print(
    f"Thời gian kiểm tra (giờ Việt Nam): {checked_times.min():%d/%m/%Y %H:%M:%S} "
    f"đến {checked_times.max():%d/%m/%Y %H:%M:%S}."
)

Kết quả,Số tin,Tỷ lệ (%)
Lấy được mô tả,10547,70.31
API trả HTTP 404/410,4453,29.69
Mô tả rỗng,0,0.00
Tin không hoạt động,0,0.00
Chưa kiểm tra,0,0.00
Lỗi HTTP khác,0,0.00
Lỗi mạng,0,0.00
Phản hồi không hợp lệ,0,0.00
Giới hạn tốc độ,0,0.00


Độ phủ mô tả: 10,547/15,000 tin (70.31%).
Thời gian kiểm tra (giờ Việt Nam): 28/09/2026 10:13:58 đến 28/09/2026 11:19:24.
